In [ ]:
from opfython.models import SupervisedOPF
from io import StringIO
from contextlib import redirect_stdout, redirect_stderr
import logging
    

In [3]:
from testflows.combinatorics import Covering
from matplotlib.ticker import MaxNLocator
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
from sklearn import svm
import random
from sklearn.neighbors import KNeighborsClassifier
from numpy import inf
import time
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score
import umap
from joblib import Parallel, delayed
import logging
import sys
import warnings
import os
from contextlib import redirect_stdout, redirect_stderr

# Suppress all logging output
os.environ['OPF_LOG_LEVEL'] = 'ERROR'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['PYTHONWARNINGS'] = 'ignore'

warnings.filterwarnings('ignore')

# Suppress all loggers - disable INFO and DEBUG
logging.getLogger().setLevel(logging.WARNING)
logging.disable(logging.INFO)

# Disable specific loggers
for logger_name in ['opfython', 'opfython.core', 'opfython.models', 'opfython.ml']:
    logger = logging.getLogger(logger_name)
    logger.setLevel(logging.WARNING)
    logger.disabled = True
    logger.propagate = False
    logger.handlers.clear()
    logger.addHandler(logging.NullHandler())

/opt/miniconda3/envs/your_env_name/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
df_cis = pd.read_csv('../data/cis_data.csv')
df_cacao = pd.read_csv(r'../data/data_temp/cacao.csv')
df_algarrobo = pd.read_csv(r'../data/data_temp/algarrobo.csv')
df_fruits_pures = pd.read_csv(r'../data/data_temp/MIR_Fruit_purees.csv')
df_fresh_meat = pd.read_csv(r'../data/data_temp/Fresh_meats.csv')
df_olive = pd.read_csv(r'../data/data_temp/Olive_Oils_Quadrum.csv')


df_x_cacao = df_cacao.iloc[:, 1:]
y_cacao = df_cacao.iloc[:, 0:1]
X_cacao = (df_x_cacao-df_x_cacao.min())/(df_x_cacao.max()-df_x_cacao.min())

algarrobo_x = df_algarrobo.loc[:, 'R':'REDVI']
y_algarrobo = df_algarrobo['Labels'].replace(to_replace=['N', 'P'], value=[0, 1])
X_algarrobo = (algarrobo_x-algarrobo_x.min())/(algarrobo_x.max()-algarrobo_x.min())

cis_x = df_cis[['X', 'Y', 'X10', 'Y10', 'X20', 'Y20', 'X30', 'Y30', 'X40', 'Y40']]
y_cis = df_cis[['Result']]
X_cis = (cis_x-cis_x.min())/(cis_x.max()-cis_x.min())

unique_names_berry = df_fruits_pures["label"].unique()
fruits_pures_x = df_fruits_pures.iloc[:,1:]
y_fruit_puree = df_fruits_pures.iloc[:,0:1].replace(to_replace=unique_names_berry, value =range(0,len(unique_names_berry)))
X_fruit_puree = (fruits_pures_x-fruits_pures_x.min())/(fruits_pures_x.max()-fruits_pures_x.min())

unique_names_meat = df_fresh_meat["meat"].unique()
meat_x = df_fresh_meat.iloc[:,4:]
y_meat = df_fresh_meat.iloc[:,0:1].replace(to_replace=unique_names_meat,value=range(0,len(unique_names_meat)))
X_meat =  (meat_x-meat_x.min())/(meat_x.max()-meat_x.min())

unique_names_olive = df_olive["Provenance"].unique()
olive_x = df_olive.iloc[:,3:]
y_olive = df_olive.iloc[:,2:3].replace(to_replace=unique_names_olive,value=range(0,len(unique_names_olive)))
X_olive =  (olive_x-olive_x.min())/(olive_x.max()-olive_x.min())

covering_array  = np.loadtxt(r'../data/coveringArray.csv', delimiter=",", dtype=int)

In [5]:
from sklearn.model_selection import KFold

def cafs(ca,dataset_x,dataset_y, max_iter ,clasifier,print_logs=False):

  global_max = 0
  max_iteartion = 0
  num_rows = ca.shape[0]
  global_data_set = dataset_x.columns.tolist()
  
  std_list = []
  result_list_x = []
  result_list_y = []
  result_list_score = []
  
  if len(dataset_x.columns) <= ca.shape[1] :
    num_colums = len(dataset_x.columns)
  else:
    num_colums = ca.shape[1]

  #initialTestTraining(result_list_score,result_list_x, result_list_y,dataset_x, dataset_y,clasifier)
  while max_iteartion < max_iter:

     lst_subset_of_candidate_features = []
     lst_headers = global_data_set.copy()
     
     best_std = 0.0
     max_score = 0.0
     mx_data_set = None

     for i in range(0,num_rows):
        lst_headers_to_select = []
        for j in range(0,num_colums ):
          if ca[i][j] == 1 :
              lst_headers_to_select.append(lst_headers[j])
        #with the list of headers to select get sub dat set of col with pandas
        if len(lst_headers_to_select) == 0:
            continue
        #df_temp = dataset_x[lst_headers_to_select]
        lst_subset_of_candidate_features.append((lst_headers_to_select,i))
    
     results = Parallel(n_jobs=-1, backend='loky')(delayed(run_cv)(clasifier, dataset_x, subset_features, dataset_y.values.ravel(), i) for subset_features,i in lst_subset_of_candidate_features)
     sorted_results = sorted(results, key=lambda x: x[2])
     for score, std, i, subset_features in sorted_results:
        if score >= max_score:
              max_score = score
              best_std = std
              mx_data_set = subset_features.copy()
              
     global_data_set = mx_data_set.copy()
     global_max = max_score
     if print_logs:
         print(f"best f1 score= {global_max}, iteration:{max_iteartion}, numbers features selected ={len(global_data_set)},best features selected={', '.join(global_data_set)}" )

     num_colums  = len(global_data_set)
     mx_data_set = None
     max_score = 0
     max_iteartion = max_iteartion  +1
     result_list_x.append(max_iteartion)
     result_list_score.append(global_max)
     result_list_y.append(len(global_data_set))
     std_list.append(best_std)

  return result_list_x,result_list_y,result_list_score,std_list

def initialTestTraining(score_list, iter_list, feature_list ,X,y,clasifier):
    
    score = kfold_with_opf_in_icafs(clasifier,X.values, y.values.ravel())
    score_list.append(score.mean())
    feature_list.append(X.shape[1])
    iter_list.append(0)

def kfold_with_opf_in_icafs(clasifier,x_data, y_data):
    
    scores = []
    kf = KFold(n_splits=5,shuffle=True, random_state =42)
    for i, (train_index, test_index) in enumerate(kf.split(x_data)):
   
        x_fold_train = x_data[train_index,:]
        y_fold_train = y_data[train_index]

        x_fold_test =  x_data[test_index,:]
        y_fold_test = y_data[test_index]
        
        clasifier.fit(x_fold_train, y_fold_train)
        y_pred_fold = clasifier.predict(x_fold_test)
        score = f1_score(y_fold_test, y_pred_fold,average='macro')
        scores.append(score)

    return np.asarray(scores, dtype=np.float32)

def run_cv(model, X, subset_features, y, index):
    
    # Create null file to suppress all output
    null_file = StringIO()
    # Disable INFO and DEBUG logging temporarily
    logging.disable(logging.INFO)
    
    try:
        # Redirect both stdout and stderr to null
        with redirect_stdout(null_file), redirect_stderr(null_file):
            scores = kfold_with_opf_in_icafs(model, X[subset_features].values, y)
        return (scores.mean(), scores.std(), index, subset_features)
    finally:
        # Restore logging state
        logging.disable(logging.NOTSET)

In [6]:
def  plot_results_for_covering_array(scores,feature,num_of_iterarion, path_to_save_image):
        color = 'tab:blue'
        res_scores = np.array(scores)
        res_features = np.array(feature)
        res_iter = np.array(num_of_iterarion)

        plt.figure(figsize=(11, 10))
        
        fig, ax1 = plt.subplots()
        barwidth = 0.4
        color = 'tab:red'
        ax1.set_xlabel('Iterations')
        ax1.set_ylabel('Number of features', color=color)
        #ax1.set_title("ICAFS Feature selection on the Cacao dataset")
        ax1.spines['top'].set_visible(False)
        ax1.bar(res_iter-0.2, res_features, color=color, width=barwidth)
        ax1.tick_params(axis='y', labelcolor=color,labelrotation=45)
        ax1.set_ylim(1,max(res_features)+3)
        ax1.xaxis.set_major_locator(MaxNLocator(integer=True))
        #for i in range(len(res_iter)):
        #    ax1.text(i+1-0.2,    res_features[i], res_features[i],rotation='vertical')
        for bar in ax1.patches:
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width() / 2.0, height, f' {height}', fontsize=10,
                    ha='center', va='bottom', rotation=90)
            
        ax2 = ax1.twinx()  # instantiate a second Axes that shares the same x-axis
        color = 'tab:blue'
        ax2.set_ylabel('F1_score', color=color)
        ax2.bar(res_iter+0.2, res_scores, color=color, width=barwidth)
        ax2.tick_params(axis='y', labelcolor=color,labelrotation=45)
        ax2.set_ylim(min(res_scores)-0.001, max(res_scores)+0.001)

        fig.tight_layout()  # otherwise the right y-label is slightly clipped
        #for i in range(len(res_iter)):
        #    ax2.text(i+1, res_scores[i], f"{res_scores[i]:.3f}",rotation='vertical')

        for bar in ax2.patches:
            height = bar.get_height()
            ax2.text(bar.get_x() + bar.get_width() / 2.0, height, f' {height:.2f}', fontsize=10,
                    ha='center', va='bottom', rotation=90)

        #plt.title('ICAFS Feature selection for Cacao Dataset with OPF', y=-0.20)
        plt.gca().set_frame_on(False)
        plt.savefig(path_to_save_image)

# OPF

In [6]:
from opfython.models.supervised import SupervisedOPF

start_time = time.time()
lr_algo = SupervisedOPF(distance="log_squared_euclidean", pre_computed_distance=None)
iterarions,features,scores,std_list = cafs(covering_array,X_cacao,y_cacao,10,lr_algo,True)
#plot_results_for_covering_array(scores,features,iterarions, r'../output_images/cacao_opf_cafs.png')
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")

best f1 score= 0.9364160299301147, iteration:0, numbers features selected =703,best features selected=1100, 1102, 1103, 1104, 1106, 1107, 1108, 1109, 1112, 1113, 1114, 1117, 1118, 1120, 1121, 1122, 1123, 1125, 1127, 1128, 1132, 1135, 1142, 1143, 1145, 1147, 1148, 1150, 1151, 1155, 1157, 1158, 1159, 1160, 1161, 1167, 1170, 1171, 1173, 1175, 1178, 1179, 1183, 1186, 1187, 1188, 1189, 1190, 1191, 1192, 1193, 1194, 1195, 1197, 1198, 1199, 1200, 1201, 1203, 1206, 1208, 1209, 1212, 1213, 1219, 1220, 1221, 1222, 1223, 1224, 1225, 1226, 1228, 1229, 1230, 1232, 1236, 1238, 1239, 1242, 1244, 1245, 1246, 1247, 1249, 1251, 1252, 1255, 1259, 1260, 1261, 1262, 1263, 1264, 1265, 1269, 1273, 1276, 1277, 1280, 1281, 1284, 1285, 1286, 1287, 1289, 1291, 1292, 1298, 1301, 1302, 1304, 1307, 1309, 1310, 1311, 1312, 1315, 1316, 1317, 1319, 1326, 1327, 1330, 1331, 1332, 1336, 1337, 1338, 1340, 1345, 1347, 1349, 1350, 1352, 1354, 1356, 1357, 1358, 1359, 1363, 1367, 1368, 1369, 1371, 1373, 1374, 1375, 1376, 1379

In [7]:
print("CAFS OPF - Cacao Dataset")
for i, std in enumerate(std_list):
    print(f"Index {i}: Dataset Cacao, Std: {std:.6f}")

CAFS OPF - Cacao Dataset
Index 0: Dataset Cacao, Std: 0.012693
Index 1: Dataset Cacao, Std: 0.011917
Index 2: Dataset Cacao, Std: 0.013966
Index 3: Dataset Cacao, Std: 0.016863
Index 4: Dataset Cacao, Std: 0.019165
Index 5: Dataset Cacao, Std: 0.014314
Index 6: Dataset Cacao, Std: 0.013280
Index 7: Dataset Cacao, Std: 0.015004
Index 8: Dataset Cacao, Std: 0.020978
Index 9: Dataset Cacao, Std: 0.020978


In [8]:
start_time = time.time()
algorithm = SupervisedOPF(distance="log_squared_euclidean", pre_computed_distance=None)
iterations ,feature,score,std_list = cafs(covering_array,X_algarrobo,y_algarrobo,10,lr_algo,True)
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")
#plot_results_for_covering_array(score,feature,iterations, r'../output_images/algarrobo_opf_icafs.png')

best f1 score= 0.7659595012664795, iteration:0, numbers features selected =12,best features selected=G, EXG, NGRDI, NGBDI, GBRI.1, VEG, RGBVI, MGRVI, NDVI, DVI, REVI, NDRE
best f1 score= 0.7703493237495422, iteration:1, numbers features selected =8,best features selected=G, EXG, NGRDI, GBRI.1, VEG, MGRVI, NDVI, DVI
best f1 score= 0.7700234651565552, iteration:2, numbers features selected =5,best features selected=EXG, NGRDI, VEG, NDVI, DVI
best f1 score= 0.7700234651565552, iteration:3, numbers features selected =5,best features selected=EXG, NGRDI, VEG, NDVI, DVI
best f1 score= 0.7700234651565552, iteration:4, numbers features selected =5,best features selected=EXG, NGRDI, VEG, NDVI, DVI
best f1 score= 0.7700234651565552, iteration:5, numbers features selected =5,best features selected=EXG, NGRDI, VEG, NDVI, DVI
best f1 score= 0.7700234651565552, iteration:6, numbers features selected =5,best features selected=EXG, NGRDI, VEG, NDVI, DVI
best f1 score= 0.7700234651565552, iteration:7, 

In [9]:
print("CAFS OPF - Algarrobo Dataset")
for i, std in enumerate(std_list):
    print(f"Index {i}: Dataset Algarrobo, Std: {std:.6f}")

CAFS OPF - Algarrobo Dataset
Index 0: Dataset Algarrobo, Std: 0.033332
Index 1: Dataset Algarrobo, Std: 0.042827
Index 2: Dataset Algarrobo, Std: 0.028655
Index 3: Dataset Algarrobo, Std: 0.028655
Index 4: Dataset Algarrobo, Std: 0.028655
Index 5: Dataset Algarrobo, Std: 0.028655
Index 6: Dataset Algarrobo, Std: 0.028655
Index 7: Dataset Algarrobo, Std: 0.028655
Index 8: Dataset Algarrobo, Std: 0.028655
Index 9: Dataset Algarrobo, Std: 0.028655


In [10]:
start_time = time.time()
algorithm = SupervisedOPF(distance="log_squared_euclidean", pre_computed_distance=None)
iterations ,feature,score,std_list = cafs(covering_array,X_fruit_puree,y_fruit_puree,10,lr_algo,True)
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")
#plot_results_for_covering_array(score,feature,iterations, r'.\\output_images\\fruit_pure_opf_icafs.png')

best f1 score= 0.9602241516113281, iteration:0, numbers features selected =117,best features selected=899.327, 907.047, 910.907, 918.627, 922.487, 926.347, 930.207, 934.067, 941.787, 949.507, 957.227, 961.087, 972.667, 984.247, 988.107, 1003.547, 1007.407, 1015.127, 1030.567, 1034.427, 1038.287, 1049.867, 1053.727, 1065.307, 1073.027, 1088.467, 1092.327, 1096.187, 1103.907, 1107.767, 1123.207, 1134.786, 1146.366, 1150.226, 1157.946, 1161.806, 1169.526, 1188.826, 1196.546, 1204.266, 1215.846, 1223.566, 1227.426, 1250.586, 1254.446, 1269.886, 1277.606, 1281.466, 1296.906, 1304.626, 1308.486, 1312.346, 1323.926, 1327.786, 1331.646, 1335.506, 1339.366, 1347.086, 1354.806, 1358.666, 1366.386, 1374.106, 1381.826, 1385.686, 1389.545, 1393.405, 1401.125, 1412.705, 1424.285, 1428.145, 1435.865, 1439.725, 1447.445, 1451.305, 1455.165, 1466.745, 1482.185, 1486.045, 1489.905, 1497.625, 1505.345, 1509.205, 1520.785, 1524.645, 1540.085, 1543.945, 1547.805, 1555.525, 1567.105, 1570.965, 1574.825, 158

In [11]:
print("CAFS OPF - Fruit Puree Dataset")
for i, std in enumerate(std_list):
    print(f"Index {i}: Dataset Fruit Puree, Std: {std:.6f}")

CAFS OPF - Fruit Puree Dataset
Index 0: Dataset Fruit Puree, Std: 0.011345
Index 1: Dataset Fruit Puree, Std: 0.007436
Index 2: Dataset Fruit Puree, Std: 0.007268
Index 3: Dataset Fruit Puree, Std: 0.011717
Index 4: Dataset Fruit Puree, Std: 0.010525
Index 5: Dataset Fruit Puree, Std: 0.012000
Index 6: Dataset Fruit Puree, Std: 0.013880
Index 7: Dataset Fruit Puree, Std: 0.013880
Index 8: Dataset Fruit Puree, Std: 0.013880
Index 9: Dataset Fruit Puree, Std: 0.013880


In [12]:
start_time = time.time()
algorithm = SupervisedOPF(distance="log_squared_euclidean", pre_computed_distance=None)
iterations ,feature,score,std_list = cafs(covering_array,X_meat,y_meat,10,lr_algo,True)
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")
#plot_results_for_covering_array(score,feature,iterations, r'.\\output_images\\meat_opf_icafs.png')

best f1 score= 0.9425075650215149, iteration:0, numbers features selected =225,best features selected=1007.279, 1011.138, 1014.997, 1016.9265, 1018.856, 1020.7855, 1028.5035, 1036.2215, 1038.151, 1040.0815, 1043.9405, 1047.7995, 1049.729, 1051.6585, 1053.588, 1059.3765, 1063.2355, 1065.165, 1067.0945, 1070.9535, 1074.8135, 1078.6725, 1080.602, 1082.5315, 1084.461, 1086.3905, 1088.32, 1090.2495, 1092.179, 1094.1085, 1099.897, 1103.756, 1105.6855, 1111.475, 1113.4045, 1115.334, 1117.2635, 1119.193, 1121.1225, 1123.052, 1126.911, 1128.8405, 1132.7005, 1134.63, 1136.5595, 1142.348, 1150.066, 1151.9955, 1153.925, 1165.502, 1171.2915, 1175.1505, 1177.08, 1179.0095, 1180.939, 1190.5875, 1196.376, 1198.3055, 1200.235, 1202.1645, 1207.953, 1209.8825, 1217.6005, 1221.4595, 1223.389, 1227.249, 1229.1785, 1238.826, 1240.7555, 1242.685, 1244.6145, 1246.544, 1256.1925, 1261.981, 1263.9105, 1269.699, 1277.417, 1279.3465, 1281.276, 1283.2065, 1287.0655, 1288.995, 1292.854, 1294.7835, 1296.713, 1298.64

In [13]:
print("CAFS OPF - Meat Dataset")
for i, std in enumerate(std_list):
    print(f"Index {i}: Dataset Meat, Std: {std:.6f}")

CAFS OPF - Meat Dataset
Index 0: Dataset Meat, Std: 0.050743
Index 1: Dataset Meat, Std: 0.055908
Index 2: Dataset Meat, Std: 0.059211
Index 3: Dataset Meat, Std: 0.070985
Index 4: Dataset Meat, Std: 0.059211
Index 5: Dataset Meat, Std: 0.070987
Index 6: Dataset Meat, Std: 0.073307
Index 7: Dataset Meat, Std: 0.073307
Index 8: Dataset Meat, Std: 0.073307
Index 9: Dataset Meat, Std: 0.073307


In [14]:
start_time = time.time()
algorithm = SupervisedOPF(distance="log_squared_euclidean", pre_computed_distance=None)
iterations ,feature,score,std_list = cafs(covering_array,X_olive,y_olive,10,lr_algo,True)
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")
#plot_results_for_covering_array(score,feature,iterations, r'.\\output_images\\olive_opf_icafs.png')

best f1 score= 0.8677147030830383, iteration:0, numbers features selected =269,best features selected=798.892, 808.5395, 814.328, 816.2575, 818.187, 820.1165, 825.905, 827.8345, 835.5525, 839.4115, 841.341, 845.2, 847.1295, 849.059, 858.7065, 860.636, 862.5655, 864.495, 866.4245, 876.072, 881.8605, 883.79, 889.5785, 897.2965, 899.226, 901.1555, 903.085, 906.944, 908.8735, 912.7325, 914.662, 916.5915, 918.521, 920.4505, 924.3095, 928.1685, 932.0275, 933.957, 939.7455, 945.534, 947.4645, 955.1825, 957.112, 960.971, 968.689, 970.6185, 972.548, 978.3365, 980.266, 986.0545, 989.9135, 997.6315, 999.561, 1001.4905, 1005.3495, 1007.279, 1014.997, 1024.6445, 1026.574, 1030.433, 1032.3625, 1036.2215, 1045.87, 1049.729, 1053.588, 1059.3765, 1063.2355, 1065.165, 1076.743, 1078.6725, 1086.3905, 1090.2495, 1092.179, 1099.897, 1103.756, 1105.6855, 1107.615, 1113.4045, 1115.334, 1117.2635, 1119.193, 1121.1225, 1126.911, 1130.77, 1132.7005, 1136.5595, 1140.4185, 1144.2775, 1146.207, 1148.1365, 1151.995

In [15]:
print("CAFS OPF - Olive Dataset")
for i, std in enumerate(std_list):
    print(f"Index {i}: Dataset Olive, Std: {std:.6f}")

CAFS OPF - Olive Dataset
Index 0: Dataset Olive, Std: 0.041135
Index 1: Dataset Olive, Std: 0.044236
Index 2: Dataset Olive, Std: 0.074917
Index 3: Dataset Olive, Std: 0.043975
Index 4: Dataset Olive, Std: 0.073214
Index 5: Dataset Olive, Std: 0.086910
Index 6: Dataset Olive, Std: 0.080666
Index 7: Dataset Olive, Std: 0.078190
Index 8: Dataset Olive, Std: 0.112764
Index 9: Dataset Olive, Std: 0.112764


### Generate CA for CIS

In [6]:
from testflows.combinatorics import Covering

items = []
paramter_to_test = {"X":[0,1],"Y":[0,1],"X10":[0,1],"Y10":[0,1],"X20":[0,1],"Y20":[0,1],"X30":[0,1],"Y30":[0,1],"X40":[0,1],"Y40":[0,1]}
generate_covering_array = Covering(paramter_to_test, strength=2)
for row in generate_covering_array.array:
   items_2 =[]
   for(key,value) in row.items():
        items_2.append(value)
   items.append(items_2)
ca_for_cis =  np.array(items)
ca_for_cis

array([[0, 0, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
       [1, 0, 0, 1, 0, 1, 0, 1, 0, 1],
       [1, 1, 1, 0, 1, 0, 1, 0, 1, 0],
       [0, 0, 0, 0, 1, 1, 1, 0, 0, 1],
       [0, 1, 0, 1, 1, 1, 0, 1, 1, 0],
       [0, 0, 1, 0, 0, 0, 1, 1, 1, 1],
       [0, 0, 0, 1, 0, 0, 1, 0, 1, 0],
       [0, 0, 1, 0, 0, 0, 0, 1, 0, 1],
       [0, 1, 0, 0, 0, 0, 0, 0, 0, 1]])

In [7]:
from opfython.models.supervised import SupervisedOPF

start_time = time.time()
lr_algo = SupervisedOPF(distance="log_squared_euclidean", pre_computed_distance=None)
iterarions,features,scores,std_list = cafs(ca_for_cis,X_cis,y_cis,10,lr_algo,True)
#plot_results_for_covering_array(scores,features,iterarions, r'../output_images/cis_opf_cafs.png')
end_time = time.time()
print(f"Processing time: {end_time - start_time} seconds")

best f1 score= 0.9845951199531555, iteration:0, numbers features selected =2,best features selected=Y, Y40
best f1 score= 0.9845951199531555, iteration:1, numbers features selected =2,best features selected=Y, Y40
best f1 score= 0.9845951199531555, iteration:2, numbers features selected =2,best features selected=Y, Y40
best f1 score= 0.9845951199531555, iteration:3, numbers features selected =2,best features selected=Y, Y40
best f1 score= 0.9845951199531555, iteration:4, numbers features selected =2,best features selected=Y, Y40
best f1 score= 0.9845951199531555, iteration:5, numbers features selected =2,best features selected=Y, Y40
best f1 score= 0.9845951199531555, iteration:6, numbers features selected =2,best features selected=Y, Y40
best f1 score= 0.9845951199531555, iteration:7, numbers features selected =2,best features selected=Y, Y40
best f1 score= 0.9845951199531555, iteration:8, numbers features selected =2,best features selected=Y, Y40
best f1 score= 0.9845951199531555, it

In [8]:
print("CAFS OPF - CIS Dataset")
for i, std in enumerate(std_list):
    print(f"Index {i}: Dataset CIS, Std: {std:.6f}")

CAFS OPF - CIS Dataset
Index 0: Dataset CIS, Std: 0.002353
Index 1: Dataset CIS, Std: 0.002353
Index 2: Dataset CIS, Std: 0.002353
Index 3: Dataset CIS, Std: 0.002353
Index 4: Dataset CIS, Std: 0.002353
Index 5: Dataset CIS, Std: 0.002353
Index 6: Dataset CIS, Std: 0.002353
Index 7: Dataset CIS, Std: 0.002353
Index 8: Dataset CIS, Std: 0.002353
Index 9: Dataset CIS, Std: 0.002353
